# Fix 02 — equal windows and explicit event eligibility

Use February 1–28 as features and March 1–28 as outcomes: both windows contain four complete weeks.
Pages must have all 28 prior days available, positive prior impressions and clicks, and the
required position/metadata features. Zero prior clicks make percentage decline undefined:
those pages are outside this task, not negative cases. Low positive counts remain and are
reported in 1–4, 5–19 and 20+ prior-click bands.

An as-of eligible page with incomplete March observation has an **unknown outcome**, not zero
performance. Its exclusion from evaluation is counted. The estimand is conditional on outcome
observability; missingness is not assumed random. Joins use both page keys and validate cardinality.

A descriptive paired-label table compares the 28-day rule with the old 31-day March rule on
the same pages with full coverage of both outcome windows. This isolates label disagreements
from changing the denominator; it is not a causal experiment.

The three-model comparison and inner-validation selection are retained from Fix 01.
This is still a single retrospective client split, not a prospective temporal validation.
The original median-gap baseline may remain weak and must not establish general ML superiority.

Run in a fresh Colab runtime with your existing HF_TOKEN Secret enabled. If version verification
requests a restart, restart the runtime and Run all again. Share only aggregate JSON/PNG.


In [ ]:
import os, sys, subprocess, importlib, importlib.metadata
SMOKE = os.environ.get('FLYRANK_FIX02_SMOKE') == '1'
PINS = {'numpy':'2.2.6','pandas':'2.2.3','scipy':'1.16.3',
        'scikit-learn':'1.7.2','matplotlib':'3.10.7','duckdb':'1.4.1','huggingface_hub':'1.0.1'}
if not SMOKE:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',
                          *[f'{name}=={version}' for name,version in PINS.items()]])
    mismatches=[]
    for name,expected in PINS.items():
        module=importlib.import_module('sklearn' if name=='scikit-learn' else name)
        loaded=getattr(module,'__version__',None)
        installed=importlib.metadata.version(name)
        if loaded!=expected or installed!=expected:
            mismatches.append({'package':name,'loaded':loaded,'installed':installed,'expected':expected})
    if mismatches:
        raise RuntimeError(f'Restart the Colab runtime, then Run all again. Loaded libraries do not match installed pins: {mismatches}')
print('SYNTHETIC CHECK ONLY' if SMOKE else 'Warehouse mode; runtime versions verified')


In [ ]:
"""Equal-window event measurement; outcome availability is distinct from a negative label."""
import numpy as np
import pandas as pd

KEYS = ['client_hash_id', 'content_hash_id']


def repaired_label(prior_impressions, prior_clicks, outcome_impressions, outcome_clicks, observed):
    """Exact integer threshold comparisons; undefined or unobserved events stay NA."""
    for series in [prior_impressions,prior_clicks,outcome_impressions,outcome_clicks]:
        values=series.dropna()
        if not np.isfinite(values).all() or (values<0).any() or (values%1!=0).any():
            raise ValueError('Count metrics must be finite, nonnegative integers when present.')
    index=prior_impressions.index
    result=pd.Series(pd.NA,index=index,dtype='Int64')
    known=(observed & prior_impressions.gt(0) & prior_clicks.gt(0)
           & outcome_impressions.notna() & outcome_clicks.notna())
    pi=prior_impressions.loc[known].astype('int64')
    pc=prior_clicks.loc[known].astype('int64')
    oi=outcome_impressions.loc[known].astype('int64')
    oc=outcome_clicks.loc[known].astype('int64')
    result.loc[known]=((10*oi>=9*pi)&(10*oi<=11*pi)&(20*oc<=17*pc)).astype('int64')
    return result


def prepare_equal_windows(feb, mar, metadata):
    """Inputs are page-level aggregates with counts of observed and available dates."""
    for name,frame in [('February',feb),('March',mar),('metadata',metadata)]:
        if frame.duplicated(KEYS).any():
            raise ValueError(f'{name}: duplicate page keys; audit before merging.')
    for name,frame in [('February',feb),('March',mar)]:
        if (frame.raw_rows != frame.raw_days).any():
            raise ValueError(f'{name}: duplicate page-day rows; do not sum them twice.')
        if (frame.bad_counts > 0).any():
            raise ValueError(f'{name}: available rows with null/negative count metrics.')
    d=feb.merge(metadata,on=KEYS,how='left',validate='one_to_one')
    # A missing outcome is missing; counts are never replaced with zero here.
    d=d.merge(mar.drop(columns=['raw_rows','raw_days','bad_counts']),on=KEYS,
              how='left',validate='one_to_one')
    full_feb=d.available_days_prior28.eq(28)
    positive_impressions=d.impressions_prior28.gt(0)
    positive_clicks=d.clicks_prior28.gt(0)
    supported_metadata=d[['content_type','main_intent']].notna().all(axis=1)
    supported_position=d.avg_position_prior28.notna()&d.avg_position_prior28.gt(0)
    stages=[('all_february_pages',pd.Series(True,index=d.index)),
            ('complete_february_coverage',full_feb),
            ('positive_prior_impressions',full_feb&positive_impressions),
            ('positive_prior_clicks',full_feb&positive_impressions&positive_clicks),
            ('required_features_available',full_feb&positive_impressions&positive_clicks
                                          &supported_metadata&supported_position)]
    flow=[{'stage':name,'n_pages':int(mask.sum()),
           'n_clients':int(d.loc[mask,'client_hash_id'].nunique())} for name,mask in stages]
    eligible=stages[-1][1]
    observed28=d.available_days_outcome28.eq(28)
    observed31=d.available_days_outcome31.eq(31)
    d['ctr_prior28']=100*d.clicks_prior28/d.impressions_prior28.replace(0,np.nan)
    d['decoupling_signature']=repaired_label(d.impressions_prior28,d.clicks_prior28,
        d.impressions_outcome28,d.clicks_outcome28,eligible&observed28)
    d['calendar31_control_label']=repaired_label(d.impressions_prior28,d.clicks_prior28,
        d.impressions_outcome31,d.clicks_outcome31,eligible&observed31)
    evaluation=d.loc[eligible&observed28].copy()
    evaluation['decoupling_signature']=evaluation.decoupling_signature.astype('int64')
    paired=d.loc[eligible&observed28&observed31]
    label_table=[]
    for a in (0,1):
        for b in (0,1):
            label_table.append({'equal28_label':a,'unequal31_label':b,
                'n_pages':int(((paired.decoupling_signature==a)&
                               (paired.calendar31_control_label==b)).sum())})
    buckets=pd.cut(evaluation.clicks_prior28,bins=[0,4,19,np.inf],
                   labels=['1-4','5-19','20+'])
    support=[]
    for band in ['1-4','5-19','20+']:
        part=evaluation.loc[buckets==band]
        support.append({'prior_click_band':band,'n_pages':len(part),
                        'positive_count':int(part.decoupling_signature.sum()),
                        'base_rate':float(part.decoupling_signature.mean()) if len(part) else None})
    audit={'feature_window':'2026-02-01 through 2026-02-28 inclusive',
        'outcome_window':'2026-03-01 through 2026-03-28 inclusive',
        'calendar_control_window':'2026-03-01 through 2026-03-31 inclusive',
        'as_of_eligibility':'28 available prior days; prior impressions>0 and clicks>0; required features present; positive position',
        'outcome_observability':'28 available outcome days; missing outcome is not a negative',
        'flow':flow,'n_as_of_eligible':int(eligible.sum()),
        'n_eligible_without_complete_outcome':int((eligible&~observed28).sum()),
        'n_evaluation_pages':len(evaluation),
        'n_zero_prior_clicks_with_full_prior_exposure':int((full_feb&positive_impressions&d.clicks_prior28.eq(0)).sum()),
        'paired_calendar_comparison_n':len(paired),'paired_label_counts':label_table,
        'prior_click_support':support,
        'measurement_note':'Low positive counts are retained in the main task and reported by support band; they are not claimed reliable business losses.',
        'scope_note':'Evaluation conditions on complete outcome observation; coverage losses are disclosed, not assumed random.'}
    return evaluation,audit


In [ ]:
"""Stage 2: compare all candidates; select only within the outer training clients.

The calling notebook supplies the repaired equal-window, eligible-page outcome.
Results remain exploratory until the separate temporal-validation and uncertainty repairs.
"""
from __future__ import annotations

import hashlib
import platform

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ['impressions_prior28', 'clicks_prior28', 'ctr_prior28', 'avg_position_prior28']
CATEGORICAL = ['content_type', 'main_intent']
FEATURES = NUMERIC + CATEGORICAL
KEYS = ['client_hash_id', 'content_hash_id']
TARGET = 'decoupling_signature'
CANDIDATES = ['Logistic Regression', 'Random Forest']
PRIMARY_K = 50


def make_model(name, seed=42, n_trees=300):
    # Each pipeline owns its transformer. Fitting one candidate cannot mutate another.
    prep = ColumnTransformer([
        ('num', StandardScaler(), NUMERIC),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
    ])
    if name == 'Logistic Regression':
        clf = LogisticRegression(max_iter=1000, random_state=seed)
    elif name == 'Random Forest':
        clf = RandomForestClassifier(n_estimators=n_trees, random_state=seed, n_jobs=-1)
    else:
        raise ValueError(f'Unknown candidate: {name}')
    return Pipeline([('prep', prep), ('clf', clf)])


def rank_metrics(scores, labels, ks=(25, 50, 100, 200)):
    scores, labels = np.asarray(scores, dtype=float), np.asarray(labels, dtype=int)
    if len(scores) != len(labels) or not len(scores):
        raise ValueError('Scores and labels must be nonempty and aligned.')
    if not np.isfinite(scores).all() or set(np.unique(labels)) != {0, 1}:
        raise ValueError('Finite scores and both binary outcome classes are required.')
    if max(ks) > len(labels):
        raise ValueError('Not enough evaluation rows for every declared K.')
    # Caller canonically orders page keys. Stable ties follow those keys, never the label.
    order = np.argsort(-scores, kind='stable')
    prevalence = float(labels.mean())
    out = {'n_eval': len(labels), 'positive_count': int(labels.sum()),
           'base_rate': prevalence, 'roc_auc': float(roc_auc_score(labels, scores)),
           'average_precision': float(average_precision_score(labels, scores)),
           'unique_score_count': int(np.unique(scores).size),
           'zero_score_fraction': float((scores == 0).mean())}
    for k in ks:
        hits = int(labels[order[:k]].sum())
        cutoff = scores[order[k-1]]
        above = int((scores > cutoff).sum())
        at = scores == cutoff
        remaining = k-above
        tied_hits = int(labels[at].sum())
        tied_count = int(at.sum())
        above_hits = int(labels[scores > cutoff].sum())
        out.update({f'hits@{k}': hits, f'precision@{k}': hits/k,
                    f'recall@{k}': hits/int(labels.sum()),
                    f'lift@{k}': hits/k/prevalence,
                    f'cutoff_tie_count@{k}': tied_count,
                    # Expected performance under uniformly random cutoff tie breaking.
                    f'tie_expected_precision@{k}':
                        (above_hits + remaining*tied_hits/tied_count)/k,
                    f'tie_min_precision@{k}':
                        (above_hits + max(0, remaining-(tied_count-tied_hits)))/k,
                    f'tie_max_precision@{k}':
                        (above_hits + min(remaining, tied_hits))/k})
    return out


def choose_candidate(validation_table, primary_k=PRIMARY_K):
    """Fixed simplicity preference on exact ties; never use the outer test table."""
    learned = validation_table.set_index('method').loc[CANDIDATES]
    scores = learned[f'precision@{primary_k}']
    return next(name for name in CANDIDATES if scores[name] == scores.max())


def fit_reference(train, evaluation):
    """Original median-gap rule, fitted on training rows; diagnostics expose collapse."""
    bins = [0, 3, 10, 20, 50, np.inf]
    train_bucket = pd.cut(train.avg_position_prior28, bins, include_lowest=True)
    eval_bucket = pd.cut(evaluation.avg_position_prior28, bins, include_lowest=True)
    medians = train.groupby(train_bucket, observed=False).ctr_prior28.median()
    reference = eval_bucket.map(medians).astype(float)
    if reference.isna().any():
        raise ValueError('Baseline has an unsupported/missing position bucket; audit it.')
    visible = evaluation.impressions_prior28 >= train.impressions_prior28.median()
    return (visible * (reference-evaluation.ctr_prior28).clip(lower=0)).to_numpy()


def compare_models(frame, seed=42, n_trees=300, ks=(25, 50, 100, 200)):
    if PRIMARY_K not in ks:
        raise ValueError('The prespecified primary K=50 must be evaluated.')
    d = frame.sort_values(KEYS, kind='stable').reset_index(drop=True).copy()
    if d.duplicated(KEYS).any():
        raise ValueError('Page keys are not unique; audit the joins before fitting.')
    required = FEATURES + KEYS + [TARGET]
    if d[required].isna().any().any():
        raise ValueError('Missing required values; retain a documented cohort policy.')
    outer = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=seed)
    outer_train_idx, test_idx = next(outer.split(d, groups=d.client_hash_id))
    outer_train, test = d.iloc[outer_train_idx].copy(), d.iloc[test_idx].copy()
    inner = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=seed+1)
    inner_fit_idx, val_idx = next(inner.split(outer_train, groups=outer_train.client_hash_id))
    fit, validation = outer_train.iloc[inner_fit_idx], outer_train.iloc[val_idx]
    groups = [set(x.client_hash_id) for x in (fit, validation, test)]
    if any(groups[i] & groups[j] for i in range(3) for j in range(i+1, 3)):
        raise AssertionError('Client overlap between fit, validation, and test.')
    for part in (fit, validation, test):
        if part[TARGET].nunique() != 2 or len(part) < max(ks):
            raise ValueError('Each split requires both classes and enough rows for K.')

    def evaluate(train, evaluation):
        models, scores = {}, {'Baseline rule': fit_reference(train, evaluation)}
        for name in CANDIDATES:
            model = make_model(name, seed=seed, n_trees=n_trees)
            model.fit(train[FEATURES], train[TARGET])
            models[name] = model
            scores[name] = model.predict_proba(evaluation[FEATURES])[:, 1]
        rows = [{'method': name, **rank_metrics(s, evaluation[TARGET], ks)}
                for name, s in scores.items()]
        return pd.DataFrame(rows), models, scores

    validation_table, _, _ = evaluate(fit, validation)
    selected = choose_candidate(validation_table)
    # Freeze selected before any outer test metrics are calculated.
    test_table, fitted, test_scores = evaluate(outer_train, test)
    membership = d[KEYS].copy()
    membership['split'] = 'inner_fit'
    membership.loc[outer_train.index[val_idx], 'split'] = 'inner_validation'
    membership.loc[test.index, 'split'] = 'outer_test'
    cohort_hash = hashlib.sha256(pd.util.hash_pandas_object(
        d[KEYS+FEATURES+[TARGET]], index=False).values.tobytes()).hexdigest()
    manifest = {
        'status': 'exploratory_stage_1_not_publication_validation',
        'primary_metric': 'precision@50',
        'selection_data': 'inner client validation only',
        'selection_tie_rule': 'prefer Logistic Regression',
        'ranking_tie_rule': 'canonical client/content key order, stable sort',
        'selected_method': selected, 'seed': seed, 'inner_seed': seed+1,
        'n_trees': n_trees, 'cohort_sha256': cohort_hash,
        'n_rows': len(d), 'n_clients': int(d.client_hash_id.nunique()),
        'partitions': {name: {'n_rows':len(part),
                             'n_clients':int(part.client_hash_id.nunique()),
                             'base_rate':float(part[TARGET].mean())}
                       for name,part in [('inner_fit',fit),('inner_validation',validation),
                                         ('outer_train',outer_train),('outer_test',test)]},
        'environment': {'python':platform.python_version(), 'numpy':np.__version__,
                        'pandas':pd.__version__, 'scikit_learn':sklearn.__version__},
        'known_pending_fixes': ['unequal calendar windows', 'zero-click eligibility',
                               'baseline validity', 'time-forward validation',
                               'client/time-aware uncertainty', 'prospective queue utility'],
        'note': 'Outer clients were examined in prior work; do not describe this as a new sealed test.'
    }
    return {'frame':d, 'validation':validation_table, 'test':test_table,
            'selected_method':selected, 'manifest':manifest, 'membership':membership,
            'test_frame':test, 'test_scores':test_scores, 'models':fitted}


In [ ]:
import re, json
from pathlib import Path

if SMOKE:
    rng=np.random.default_rng(23)
    n=4800
    keys=pd.DataFrame({'client_hash_id':np.repeat([f'synthetic_client_{i:02d}' for i in range(16)],300),
                       'content_hash_id':[f'synthetic_page_{i:05d}' for i in range(n)]})
    feb=keys.copy()
    feb['raw_rows']=28;feb['raw_days']=28;feb['available_days_prior28']=28;feb['bad_counts']=0
    feb['impressions_prior28']=rng.integers(100,10000,n)
    feb['clicks_prior28']=rng.integers(0,100,n)
    feb['avg_position_prior28']=rng.uniform(1,70,n)
    mar=keys.copy()
    mar['raw_rows']=31;mar['raw_days']=31;mar['available_days_outcome28']=28
    mar['available_days_outcome31']=31;mar['bad_counts']=0
    mar['impressions_outcome28']=np.maximum(0,(feb.impressions_prior28*rng.uniform(.7,1.3,n)).astype('int64'))
    mar['clicks_outcome28']=np.maximum(0,(feb.clicks_prior28*rng.uniform(.5,1.5,n)).astype('int64'))
    mar['impressions_outcome31']=(mar.impressions_outcome28*31/28).astype('int64')
    mar['clicks_outcome31']=(mar.clicks_outcome28*31/28).astype('int64')
    meta=keys.copy()
    meta['content_type']=rng.choice(['a','b'],n)
    meta['main_intent']=rng.choice(['informational','commercial'],n)
else:
    import duckdb
    token=os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token=userdata.get('HF_TOKEN')
        except Exception:
            token=None
    if not token:
        raise RuntimeError('Enable this notebook access to HF_TOKEN in Colab Secrets; do not paste it into code or chat.')
    if not re.fullmatch(r'hf_[A-Za-z0-9]+',token):
        raise RuntimeError('Expected a Hugging Face read token in Secrets.')
    con=duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")
    del token
    REL='hf://datasets/FlyRank/internship-warehouse'
    feb=con.sql(f"""
      SELECT client_hash_id,content_hash_id,
        COUNT(*) AS raw_rows,COUNT(DISTINCT report_date) AS raw_days,
        COUNT(DISTINCT report_date) FILTER(WHERE gsc_data_available IS TRUE) AS available_days_prior28,
        COUNT(*) FILTER(WHERE gsc_data_available IS TRUE AND
          (gsc_impressions IS NULL OR gsc_clicks IS NULL OR gsc_impressions<0 OR gsc_clicks<0)) AS bad_counts,
        CAST(SUM(gsc_impressions) FILTER(WHERE gsc_data_available IS TRUE) AS BIGINT) AS impressions_prior28,
        CAST(SUM(gsc_clicks) FILTER(WHERE gsc_data_available IS TRUE) AS BIGINT) AS clicks_prior28,
        SUM(gsc_sum_position) FILTER(WHERE gsc_data_available IS TRUE) /
          NULLIF(SUM(gsc_impressions) FILTER(WHERE gsc_data_available IS TRUE),0) AS avg_position_prior28
      FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')
      WHERE report_date>=DATE '2026-02-01' AND report_date<DATE '2026-03-01'
      GROUP BY 1,2
    """).df()
    mar=con.sql(f"""
      SELECT client_hash_id,content_hash_id,
        COUNT(*) AS raw_rows,COUNT(DISTINCT report_date) AS raw_days,
        COUNT(DISTINCT report_date) FILTER(WHERE gsc_data_available IS TRUE AND report_date<=DATE '2026-03-28') AS available_days_outcome28,
        COUNT(DISTINCT report_date) FILTER(WHERE gsc_data_available IS TRUE) AS available_days_outcome31,
        COUNT(*) FILTER(WHERE gsc_data_available IS TRUE AND
          (gsc_impressions IS NULL OR gsc_clicks IS NULL OR gsc_impressions<0 OR gsc_clicks<0)) AS bad_counts,
        CAST(SUM(gsc_impressions) FILTER(WHERE gsc_data_available IS TRUE AND report_date<=DATE '2026-03-28') AS BIGINT) AS impressions_outcome28,
        CAST(SUM(gsc_clicks) FILTER(WHERE gsc_data_available IS TRUE AND report_date<=DATE '2026-03-28') AS BIGINT) AS clicks_outcome28,
        CAST(SUM(gsc_impressions) FILTER(WHERE gsc_data_available IS TRUE) AS BIGINT) AS impressions_outcome31,
        CAST(SUM(gsc_clicks) FILTER(WHERE gsc_data_available IS TRUE) AS BIGINT) AS clicks_outcome31
      FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
      WHERE report_date>=DATE '2026-03-01' AND report_date<DATE '2026-04-01'
      GROUP BY 1,2
    """).df()
    meta=con.sql(f"""SELECT client_hash_id,content_hash_id,content_type,main_intent
                     FROM read_parquet('{REL}/dim_content.parquet')""").df()
    con.close()

df,cohort_audit=prepare_equal_windows(feb,mar,meta)
print('MEASUREMENT AND COHORT AUDIT')
print(json.dumps(cohort_audit,indent=2))


In [ ]:
result = compare_models(df, n_trees=12 if SMOKE else 300)
columns = ['method','hits@50','precision@50','recall@50','average_precision','roc_auc','base_rate']
print('INNER VALIDATION — learned-candidate selection by precision@50')
display(result['validation'][columns])
print('Selected learned candidate:', result['selected_method'])
print('OUTER TEST — all methods, same rows; selection already frozen')
display(result['test'][columns])
display(result['test'][['method','unique_score_count','zero_score_fraction',
    'cutoff_tie_count@50','tie_min_precision@50','tie_expected_precision@50','tie_max_precision@50']])
print('No production recommendation is made in Stage 2. Baseline validity, low-count noise and chronological validation remain.')


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve

out = Path('work/outputs/fix02_smoke' if SMOKE else 'work/outputs/fix02')
out.mkdir(parents=True, exist_ok=True)
manifest = result['manifest']
manifest['mode'] = 'synthetic_software_check' if SMOKE else 'warehouse_exploratory'
manifest['historical_target_preserved'] = False
manifest['status']='exploratory_stage_2_measurement_repair'
manifest['completed_measurement_repairs']=['equal 28-day windows','positive-prior-click eligibility','explicit missing-outcome handling','page-day coverage checks']
manifest['known_pending_fixes']=['baseline validity','low-count noise sensitivity','time-forward validation','client/time-aware uncertainty','historical metadata availability','prospective queue utility']
manifest['runtime_pins']=PINS
manifest['cohort_audit']=cohort_audit
payload = {'manifest':manifest,
    'validation':result['validation'].to_dict(orient='records'),
    'outer_test':result['test'].to_dict(orient='records')}
with (out/'model_comparison.json').open('w',encoding='utf-8') as f:
    json.dump(payload,f,indent=2,allow_nan=False)

fig, axes = plt.subplots(1,3,figsize=(15,4))
for name,scores in result['test_scores'].items():
    labels = result['test_frame'][TARGET]
    fpr,tpr,_ = roc_curve(labels,scores)
    precision,recall,_ = precision_recall_curve(labels,scores)
    axes[0].plot(fpr,tpr,label=name)
    axes[1].plot(recall,precision,label=name)
    row = result['test'].set_index('method').loc[name]
    axes[2].plot([25,50,100,200],[row[f'precision@{k}'] for k in [25,50,100,200]],marker='o',label=name)
axes[0].plot([0,1],[0,1],'k--',alpha=.4)
axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='ROC — all methods')
for ax in axes[1:]:
    ax.axhline(result['test_frame'][TARGET].mean(),color='grey',linestyle='--')
axes[1].set(xlabel='Recall',ylabel='Precision',title='Precision–recall — all methods')
axes[2].set(xlabel='Review budget K',ylabel='Precision@K',title='Review-budget performance')
for ax in axes: ax.legend(fontsize=7)
fig.suptitle('SYNTHETIC SOFTWARE CHECK — NOT RESEARCH RESULTS' if SMOKE else 'Stage 2: equal-window measurement — time validation pending')
fig.tight_layout()
fig.savefig(out/'model_comparison.png',dpi=150)
plt.show()
print('Saved aggregate metrics and figure to',out)
print('No tokens, page-level data, or client identifiers are exported.')


## Before interpreting the result

Check cohort retention and missing-outcome counts before comparing precision values with Fix 01.
The cohort and target changed, so a score change is not automatically a model improvement.
Read the paired-calendar label table and the low-click support table. Preserve every comparator.
The output is `work/outputs/fix02/model_comparison.json` and `model_comparison.png`.
No raw pages, client identifiers, or credentials are exported.
